# 06 — Approximate circuit simulation with matrix product states (MPS)

**Goal.** Simulate Trotterized interacting-qubit dynamics beyond the reach of exact state-vector simulation, and quantify the accuracy/cost trade-off.

**Model.** Transverse-field Ising chain, first-order Trotter circuit:
$$H = J\sum_{i} Z_iZ_{i+1} + h\sum_i X_i,\qquad e^{-iHt}\approx\Big[\prod_{\langle i,i+1\rangle} e^{-iJ\,\delta t\,Z_iZ_{i+1}}\prod_i e^{-ih\,\delta t\,X_i}\Big]^{t/\delta t}$$

**MPS.**
$$|\psi\rangle=\sum_{s_1\dots s_n}A^{s_1}_{[1]}A^{s_2}_{[2]}\cdots A^{s_n}_{[n]}\,|s_1\dots s_n\rangle,\qquad A^{s_i}_{[i]}\in\mathbb{C}^{\chi_{i-1}\times\chi_i}$$
Memory is $O(n\,d\,\chi^2)$ instead of $O(d^n)$. A two-qubit gate is applied by contracting neighbouring tensors, applying the gate, and splitting with an SVD truncated to $\chi_{\max}$. The discarded weight $\varepsilon=\sum_{\alpha>\chi_{\max}}\lambda_\alpha^2$ is the infidelity added by that truncation.

**Studies in this notebook**
1. Validate MPS against exact state-vector simulation (12 qubits).
2. Fidelity vs. bond dimension $\chi_{\max}$: true vs. estimated fidelity.
3. Entanglement growth and the bond dimension it requires.
4. Scaling with qubit count at fixed depth: runtime and memory.
5. Scaling with circuit depth at fixed qubit count, which is where MPS breaks down.
6. Projected ensembles (GQS) by MPS sampling: validation against the exact `gqs.states` result at 12 qubits.
7. Projected ensembles at 50–100 qubits.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from gqs.mps import (MPS, StateVector, trotter_step, prepare_initial, X, Y, Z,
                     sample_projected_ensemble, bloch_vectors, ensemble_moments,
                     fubini_study_matrix, empirical_ensemble)
from gqs.states import Reduced_state_single_site
try:
    import ot   # POT, for Wasserstein distances
except ImportError:
    ot = None

J, h, dt = 1.0, 1.0, 0.1
def run(sim, n, steps):
    prepare_initial(sim, n)
    for _ in range(steps):
        trotter_step(sim, n, J, h, dt)
    return sim

## 1. Validation against exact state-vector simulation

In [ ]:
n, steps = 12, 30
exact = run(StateVector(n), n, steps).vector()
mps_full = run(MPS(n, chi_max=None), n, steps)       # no truncation
F = abs(np.vdot(exact, mps_full.to_statevector()))**2
print(f"Fidelity (untruncated MPS vs exact): {F:.12f}")
print("Bond dimensions:", mps_full.bond_dims())

# Local observable check: <X_3>
sv = StateVector(n); sv.psi = exact.reshape((2,)*n).copy(); sv.apply_1q(X, 3)
x3_exact = np.real(np.vdot(exact, sv.vector()))
print(f"<X_3>  exact = {x3_exact:.10f}   MPS = {mps_full.expect_1q(X, 3):.10f}")

## 2. Fidelity vs. bond dimension
True fidelity $|\langle\psi_{\rm exact}|\psi_{\rm MPS}\rangle|^2$ compared with the estimate $\prod_k(1-\varepsilon_k)$, which needs no exact reference. The estimate is what lets you judge accuracy beyond the classically checkable regime. It is usually optimistic when truncation is heavy.

In [ ]:
chis = [2, 4, 8, 16, 32, 64]
true_F, est_F = [], []
for chi in chis:
    m = run(MPS(n, chi_max=chi), n, steps)
    true_F.append(abs(np.vdot(exact, m.to_statevector()))**2)
    est_F.append(m.estimated_fidelity())

plt.figure(figsize=(5, 3.5))
plt.semilogx(chis, true_F, "o-", base=2, label="true fidelity")
plt.semilogx(chis, est_F, "s--", base=2, label=r"estimate $\prod(1-\varepsilon_k)$")
plt.xlabel(r"$\chi_{\max}$"); plt.ylabel("fidelity"); plt.title(f"n={n}, {steps} Trotter steps")
plt.legend(); plt.tight_layout(); plt.show()

## 3. Entanglement growth sets the required bond dimension
Across a cut, $S=-\sum_\alpha\lambda_\alpha^2\log_2\lambda_\alpha^2\le\log_2\chi$. After a quench, $S$ grows roughly linearly in time, so the bond dimension needed for fixed accuracy grows exponentially with depth.

For $n=12$ the bond dimension saturates at its maximum $2^{n/2}=64$, and the entropy stops growing once it reaches the finite-size limit. In larger chains the growth continues, which is what drives the breakdown in Section 5.

In [ ]:
n = 12
m = MPS(n, chi_max=None); prepare_initial(m, n)
S_t, chi_t = [], []
for _ in range(60):
    trotter_step(m, n, J, h, dt)
    S_t.append(m.entropy(n // 2 - 1)); chi_t.append(max(m.bond_dims()))

t = dt * np.arange(1, 61)
fig, ax = plt.subplots(1, 2, figsize=(9, 3.2))
ax[0].plot(t, S_t); ax[0].set_xlabel("t"); ax[0].set_ylabel("half-chain entropy (bits)")
ax[1].semilogy(t, chi_t); ax[1].set_xlabel("t"); ax[1].set_ylabel(r"max bond dimension (exact)")
plt.tight_layout(); plt.show()

## 4. Scaling with qubit count (fixed depth)
At fixed depth the entanglement is bounded, so cost grows only linearly in $n$: about $O(n\,\chi^3)$ per Trotter step. A state vector needs $16\cdot2^n$ bytes.

In [ ]:
steps, chi = 30, 32
Ns = [10, 20, 40, 60, 80, 100]
runtime, mem_mps, fid = [], [], []
for N in Ns:
    t0 = time.perf_counter()
    m = run(MPS(N, chi_max=chi), N, steps)
    runtime.append(time.perf_counter() - t0)
    mem_mps.append(sum(A.nbytes for A in m.tensors))
    fid.append(m.estimated_fidelity())

mem_sv = [16 * 2.0**N for N in Ns]
fig, ax = plt.subplots(1, 2, figsize=(9, 3.2))
ax[0].plot(Ns, runtime, "o-"); ax[0].set_xlabel("qubits"); ax[0].set_ylabel("runtime (s)")
ax[1].semilogy(Ns, mem_mps, "o-", label="MPS"); ax[1].semilogy(Ns, mem_sv, "s--", label="state vector")
ax[1].set_xlabel("qubits"); ax[1].set_ylabel("memory (bytes)"); ax[1].legend()
plt.tight_layout(); plt.show()
for N, F in zip(Ns, fid):
    print(f"n={N:4d}  estimated fidelity={F:.6f}")

## 5. Scaling with depth (fixed qubit count): where MPS breaks down
At fixed $\chi_{\max}$, the estimated fidelity decays once the required bond dimension exceeds $\chi_{\max}$. This regime is the "limit of classical simulability."

In [ ]:
N, max_steps = 40, 80
plt.figure(figsize=(5, 3.5))
for chi in [8, 16, 32, 64]:
    m = MPS(N, chi_max=chi); prepare_initial(m, N)
    F_t = []
    for _ in range(max_steps):
        trotter_step(m, N, J, h, dt)
        F_t.append(m.estimated_fidelity())
    plt.plot(dt * np.arange(1, max_steps + 1), F_t, label=fr"$\chi_{{\max}}$={chi}")
plt.xlabel("t (circuit depth ∝ t)"); plt.ylabel("estimated fidelity"); plt.title(f"n={N}")
plt.legend(); plt.tight_layout(); plt.show()

## 6. Projected ensembles by MPS sampling: validation at 12 qubits

Keep one site $A$ and measure every other site in the computational basis. Each outcome $z$ leaves $A$ in
$$|\psi_z\rangle=\frac{(I_A\otimes\langle z|)\,|\psi\rangle}{\sqrt{p_z}},\qquad p_z=\|(I_A\otimes\langle z|)|\psi\rangle\|^2 .$$

**Exact method** (`gqs.states.Reduced_state_single_site`): reshape the full state vector into a $2\times2^{n-1}$ matrix whose columns are the conditional states. This needs all $2^n$ amplitudes and all $2^{n-1}$ outcomes.

**MPS sampling** (`sample_projected_ensemble`): with the MPS right-canonical, draw the measured bits one at a time from
$$p(z_j\mid z_{<j})=\frac{\|L\,A_j^{z_j}\|^2}{\|L\|^2},$$
carrying the kept site as an open index in the boundary tensor $L$. Each sample costs $O(n\chi^2)$, and the samples are drawn exactly from $p_z$.

We check three things:
1. **Exactness per sample:** each sampled $p_z$ and $|\psi_z\rangle$ equals the exact `states.py` value for that $z$.
2. **Moments:** the mean Bloch vector (the RDM) and the second moment $\mathbb{E}[\mathbf r\mathbf r^T]$ converge to the exact values.
3. **Wasserstein distance** between the sampled and exact GQS shrinks as $M$ grows. It uses the same Fubini–Study cost as `Quantum_EMD`.

In [ ]:
n, steps, site = 12, 20, 5
sv = run(StateVector(n), n, steps)
chi_exact, lam_exact = Reduced_state_single_site(2, n, site, sv.vector())   # exact GQS from states.py

m12 = run(MPS(n, chi_max=None), n, steps)
t0 = time.perf_counter()
chi_s, p_s, z_s = sample_projected_ensemble(m12, site, M=20000, seed=1)
print(f"Sampled 20000 outcomes in {time.perf_counter() - t0:.2f} s")

# 1. exactness per sample
fid = np.abs(np.sum(chi_s.conj() * chi_exact[z_s], axis=1))**2
print(f"max |p_z(MPS) - lambda_E(states.py)| = {np.max(np.abs(p_s - lam_exact[z_s])):.2e}")
print(f"min fidelity of sampled vs exact conditional states = {fid.min():.12f}")

# 2. moments
mask = lam_exact > 1e-12
m1_e, m2_e = ensemble_moments(bloch_vectors(chi_exact[mask]), lam_exact[mask])
m1_s, m2_s = ensemble_moments(bloch_vectors(chi_s))
print("mean Bloch vector  exact :", np.round(m1_e, 4))
print("mean Bloch vector  sampled:", np.round(m1_s, 4))
print(f"max |second moment difference| = {np.abs(m2_e - m2_s).max():.4f}   (~1/sqrt(M) = {1/np.sqrt(20000):.4f})")

In [ ]:
# 3. Wasserstein distance between the sampled and exact GQS vs number of samples
chi_e, w_e = chi_exact[mask], lam_exact[mask] / lam_exact[mask].sum()
if ot is None:
    print("Install POT (pip install POT) to compute Wasserstein distances.")
else:
    Ms, W = [25, 50, 100, 200, 500, 1000, 2000], []
    for M in Ms:
        c, _, z = sample_projected_ensemble(m12, site, M, seed=3)
        c_u, w_u, _ = empirical_ensemble(c, z)
        W.append(ot.emd2(w_u, w_e, fubini_study_matrix(c_u, chi_e)))
    plt.figure(figsize=(5, 3.5))
    plt.loglog(Ms, W, "o-", label="sampled vs exact GQS")
    plt.loglog(Ms, W[0] * np.sqrt(Ms[0] / np.array(Ms)), "k--", lw=1, label=r"$\propto 1/\sqrt{M}$")
    plt.xlabel("samples M"); plt.ylabel("Wasserstein distance"); plt.legend(); plt.tight_layout(); plt.show()

    # Same number as the package's Quantum_EMD (small case, to confirm the cost matches)
    from gqs.distances import Quantum_EMD
    c, _, z = sample_projected_ensemble(m12, site, 50, seed=3)
    c_u, w_u, _ = empirical_ensemble(c, z)
    print(f"Quantum_EMD = {Quantum_EMD(c_u, w_u, chi_e, w_e):.6f}   vectorized = {ot.emd2(w_u, w_e, fubini_study_matrix(c_u, chi_e)):.6f}")

## 7. Projected ensembles at 50–100 qubits

The exact method is impossible here ($2^{99}$ outcomes at $n=100$). Sampling the MPS takes well under a second. Two checks keep the results honest:
- **Truncation:** the estimated MPS fidelity $\prod(1-\varepsilon_k)$ should stay close to 1. Keep $t$ before the depth breakdown seen in Section 5.
- **Consistency:** the sampled mean Bloch vector must match $\langle X\rangle,\langle Y\rangle,\langle Z\rangle$ computed directly from the MPS, within a few times $1/\sqrt{M}$.

In [ ]:
chi_max, M, steps = 32, 2000, 20
print(f"{'n':>5} {'sample time (s)':>16} {'est. MPS fidelity':>18}   mean Bloch (sampled)  vs  direct <X,Y,Z>")
for N in [20, 50, 100, 200]:
    mN = run(MPS(N, chi_max=chi_max), N, steps)
    t0 = time.perf_counter()
    c, p, z = sample_projected_ensemble(mN, N // 2, M, seed=4)
    dt_s = time.perf_counter() - t0
    m1, _ = ensemble_moments(bloch_vectors(c))
    direct = [mN.expect_1q(O, N // 2) for O in (X, Y, Z)]
    print(f"{N:5d} {dt_s:16.2f} {mN.estimated_fidelity():18.6f}   {np.round(m1, 3)}  vs  {np.round(direct, 3)}")

In [ ]:
# How the GQS of the middle qubit spreads over the Bloch sphere as the 50-qubit chain evolves
N, M = 50, 1500
fig, axes = plt.subplots(1, 4, figsize=(13, 3.4), sharey=True)
mN = MPS(N, chi_max=32); prepare_initial(mN, N)
done = 0
for ax, target in zip(axes, [2, 10, 20, 35]):
    for _ in range(target - done):
        trotter_step(mN, N, J, h, dt)
    done = target
    c, _, _ = sample_projected_ensemble(mN, N // 2, M, seed=5)
    b = bloch_vectors(c)
    ax.add_patch(plt.Circle((0, 0), 1, fill=False, color="0.6"))
    ax.scatter(b[:, 0], b[:, 2], s=3, alpha=0.3)
    ax.set_aspect("equal"); ax.set_xlim(-1.1, 1.1); ax.set_ylim(-1.1, 1.1)
    ax.set_title(f"t = {target*dt:.1f},  F≈{mN.estimated_fidelity():.3f}")
    ax.set_xlabel("x")
axes[0].set_ylabel("z")
plt.suptitle(f"Projected ensemble of qubit {N//2} (n = {N}, χ_max = 32, M = {M})", y=1.02)
plt.tight_layout(); plt.show()

**Reading the results**
- At 12 qubits, every sampled outcome reproduces the exact `states.py` probability and conditional state to machine precision. The only remaining error is statistical, and it shrinks like $1/\sqrt{M}$.
- At 50–200 qubits, sampling takes under a second. The sampled mean Bloch vector agrees with the directly computed RDM, and the estimated MPS fidelity stays near 1 at this depth.
- At early times the results for 12, 50 and 100 qubits are nearly identical, because correlations have not yet reached the chain ends (light cone). Differences appear only at later times. That is when larger system sizes become necessary, and when the MPS needs larger $\chi$.
- Watch $F\approx\prod(1-\varepsilon_k)$ in the snapshot titles. Once it drops, the sampled ensemble inherits truncation error.


## Next steps
- Swap in the Hamiltonian from the hardware-dynamics project and compare MPS with the Qiskit Aer `matrix_product_state` method.
- Add a second-order (symmetric) Trotter step.
- Compute the unitary and non-unitary deviation metrics from sampled ensembles at large $n$.
- Add noise by sampling Pauli errors (quantum trajectories) on the MPS, which links this notebook to the coherent vs. incoherent error analysis.
